**1. MASKED AUTOENCODER TRAINING**

In [ ]:
import digitalhub as dh
import pandas as pd
import matplotlib.pyplot as plt

NOME_PROGETTO = "floods"
project = dh.get_project(NOME_PROGETTO)
print(f"Progetto: {project.name}")

**SETUP PARAMETERS**

In [ ]:
# Parametri Job
job_name = "train_v8"                           # nome da passare poi a notebook moco   train_s2_v3
dataset = "Test"    
epochs = 10                                     # Test, Standard, Anomalies*
sar_to_opt = True                               # False = opt_to_sar   
mask = False                                    # True = maschera su input
mask_precentage = 0.8                           # percentuale maschera                         
mask_square_size = 16                           # lato quadratino maschera in pixel
                                                                         
#handler = train_mae   

if sar_to_opt: direction = "sar_to_opt"
else: direction = "opt_to_sar"
                                      
parametri = {
    "epochs": epochs, 
    "batch_size": 16, 
    "lr": 1e-4, 
    "weight_decay": 1e-4,      
    "patch_size": 256, 
    "n_images1": 4, "n_channels1": 2,                       # sar
    "n_images2": 4, "n_channels2": 10,                      # opt               
    "mamba": False, 
    "workers": 0,
    "job_name": job_name,
    "dataset": dataset,
    "sar_to_opt": sar_to_opt,    
    "mask": mask,                                           # True = maschera su input
    "mask_precentage": mask_precentage,                                 # percentuale maschera                         
    "mask_square_size": mask_square_size,                                 
    "patience": 20,                                    
    "min_delta": 1e-4,
    "time_debug": True                                      # time_debug = True solo per debug, = False per training
}

print(f"PARAMETRI: {parametri}")

# volume 
volumi = [
    {
        "volume_type": "ephemeral",
        "name": "volume-spazio-dati",
        "mount_path": "/data",      
        "spec": {"size": "500Gi"}   
    }
]

**BUILD ENVIRONMENT**

In [ ]:
mae_train_func = project.new_function(
    name= f'mae-Floods_{dataset}_{job_name}_{direction}_{epochs}',
    kind="python",
    python_version="PYTHON3_10",
    code_src="../src/", 
    handler="train_mae", 
    base_image="pytorch/pytorch:2.1.2-cuda11.8-cudnn8-runtime",
    requirements=["pandas==2.3.3", "numpy==1.26.4", "rasterio==1.4.4", "tqdm==4.70.0", "tifffile==2024.8.30"]
)

# .run("build") -> scarica immagine, installa requirements, copia intera cartella code_src, crea immagine docker

build = mae_train_func.run("build", wait=True)
print(f"BUILD: {build.status.state}")

**TRAINING**

In [ ]:
# action job = avvia container, esegue script, libera risorse

run_train_mae = mae_train_func.run(
    action="job", 
    parameters=parametri, 
    volumes=volumi, 
    profile="1xV100",                                       # 1x = 1 gpu
    # local_execution= True,                                
    wait=True
)

print(f"Run train_mae avviato: {run_train_mae.id}")
print(run_train_mae.status.state)
print(run_train_mae.status.message)

**PLOTS**

In [ ]:
# salvataggio log
path_s1 = project.get_artifact(f"metrics-mae_{job_name}_{direction}_{dataset}_{epochs}").download(overwrite=True)
df_s1 = pd.read_csv(path_s1)

# plot
plt.figure(figsize=(8, 5))
plt.plot(df_s1['epoch'], df_s1['train_loss'], color='blue', label='Train Loss')
plt.title(f'Training CAE SAR - {job_name}_{direction}_{dataset}_{epochs}')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.grid(True)
plt.legend()
plt.show()

